# Module 3 Exercise: Spark ↔ Warehouse Interoperability

This is the payoff slide made concrete: **one storage layer, two engines**.
Spark writes a Delta table, the Warehouse reads it through its T-SQL front end,
no ETL in between.

SQL Server analogues:

| Fabric pattern                            | SQL Server equivalent                              |
|-------------------------------------------|----------------------------------------------------|
| Spark writes Delta, Warehouse reads it    | Linked server / cross-database query               |
| `spark.write.synapsesql("db.schema.t")`   | `OPENROWSET(...)` or bulk insert via SSIS          |
| Three-part name `gold.dbo.customers`      | Three-part name `gold.dbo.customers`               |

The big difference: there is no second copy of the data. The Warehouse is
reading the same Parquet files Spark just wrote.


## 1. Install the Spark-to-Warehouse connector (once)

The `com.microsoft.spark.fabric` connector is preinstalled on Fabric runtimes;
the `%pip install` below is only needed if you later want the full
`semantic-link-labs` toolkit for control-plane tasks.


In [ ]:
%pip install semantic-link-labs


## 2. Write a Spark DataFrame into the `gold` warehouse

The `synapsesql` method is the Spark-native writer for Fabric Warehouse. Given
a three-part name (`warehouse.schema.table`) it creates the table if needed,
loads Parquet files staged in OneLake, and commits them into the Warehouse, all
without leaving the Spark session.


In [ ]:
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

df = spark.table("silver.dbo.customers")
df.write.mode("overwrite").synapsesql("gold.dbo.customers")


## 3. Read the same table back via T-SQL

Same rows, different engine. The Warehouse sees exactly what Spark wrote because
both engines read the same Delta/Parquet files under the covers.


In [ ]:
%%sql
SELECT * FROM gold.dbo.customers


## Key takeaway

Interoperability in Fabric is not a feature, it is an emergent property of the
storage model: one lake, every engine a reader, most engines a writer. The
connector code above replaces what would have been a linked-server hop, an
Integration Services package, or a bulk-copy job in a classic SQL Server stack.
